![Identidade visual da disciplina](../capa-analise-visualizacao-python-ia.png)

# IA001 — Análise e Visualização de Dados com Python e Ferramentas Assistidas por IA

## Introdução ao Folium

**Objetivo:** construir mapas interativos com marcadores, camadas e indicadores geográficos usando Folium.

**Dados:** locais com coordenadas aproximadas em Porto Alegre, observações fictícias reproduzíveis e dados locais de bairros e população do Censo 2022.

**Roteiro:** mapa básico → marcadores → círculos e linhas → camadas → agrupamentos → mapa de calor → coroplético → exportação.

## 1. Preparação do ambiente

Folium monta em Python mapas que o navegador apresenta com **Leaflet**, uma biblioteca JavaScript. O fluxo usual é criar um `Map`, adicionar objetos com `.add_to(mapa)` e exibir o mapa.

Os dados deste notebook são locais, mas os mapas-base e as bibliotecas JavaScript/CSS são carregados pela internet. Salvar o HTML **não** torna o mapa totalmente offline. Em Jupyter, marque o notebook como confiável (*Trust Notebook*) se as saídas interativas forem bloqueadas.

Os exemplos iniciais usam Folium e pandas. GeoPandas será usado somente na seção do mapa coroplético.

In [1]:
# Se necessário, descomente e execute no kernel do notebook:
# %pip install folium pandas geopandas

from pathlib import Path
from html import escape
import random

import folium
import pandas as pd
from folium.plugins import MarkerCluster, HeatMap
from IPython.display import display, FileLink

print("Folium:", folium.__version__)
print("pandas:", pd.__version__)

Folium: 0.19.5
pandas: 2.2.3


## 2. Um primeiro mapa

`location` define o centro e `zoom_start` define a aproximação inicial. `tiles` seleciona o mapa-base; `control_scale` exibe uma escala de distâncias.

**Atenção à ordem:** Folium recebe `[latitude, longitude]`. GeoJSON armazena coordenadas como `[longitude, latitude]`. As coordenadas abaixo são aproximadas e servem à demonstração.

In [2]:
CENTRO_POA = [-30.0346, -51.2177]

mapa_basico = folium.Map(
    location=CENTRO_POA, zoom_start=12,
    tiles="OpenStreetMap", control_scale=True, height=500
)
mapa_basico

## 3. Marcadores, dicas e janelas de informação

`tooltip` aparece ao passar o mouse; `popup` abre ao clicar. A cor do ícone representa a categoria do local. Os textos usados no HTML são escapados com `escape`.

**Experimente:** passe o mouse sobre um marcador, clique nele e aproxime o mapa.

In [3]:
locais = pd.DataFrame([
    ["Mercado Público", "Cultura", -30.0277, -51.2278],
    ["Casa de Cultura Mario Quintana", "Cultura", -30.0314, -51.2345],
    ["Usina do Gasômetro", "Cultura", -30.0346, -51.2418],
    ["Parque Farroupilha", "Parque", -30.0368, -51.2166],
    ["Parque Moinhos de Vento", "Parque", -30.0226, -51.2007],
], columns=["nome", "categoria", "latitude", "longitude"])

CORES = {"Cultura": "blue", "Parque": "green"}
mapa_marcadores = folium.Map(location=CENTRO_POA, zoom_start=13, height=500)
for local in locais.itertuples(index=False):
    folium.Marker(
        location=[local.latitude, local.longitude],
        tooltip=local.nome,
        popup=folium.Popup(
            f"<b>{escape(local.nome)}</b><br>Categoria: {escape(local.categoria)}",
            max_width=250
        ),
        icon=folium.Icon(color=CORES[local.categoria], icon="info-sign")
    ).add_to(mapa_marcadores)
mapa_marcadores

## 4. Círculos e linhas

- `Circle`: raio em **metros**, útil para representar uma distância aproximada no mapa.
- `CircleMarker`: raio em **pixels**, mantendo o tamanho visual durante o zoom.
- `PolyLine`: conecta coordenadas na ordem fornecida.

O círculo abaixo representa um raio ilustrativo de 500 m. A linha liga três locais diretamente: **não é uma rota calculada pelas ruas**, e o círculo não representa tempo de caminhada.

**Experimente:** altere o zoom e compare os dois tipos de círculo.

In [4]:
mapa_formas = folium.Map(location=CENTRO_POA, zoom_start=14, height=500)
for local in locais.itertuples(index=False):
    folium.CircleMarker(
        [local.latitude, local.longitude], radius=7,
        color=CORES[local.categoria], fill=True, fill_opacity=0.8,
        tooltip=local.nome
    ).add_to(mapa_formas)

folium.Circle(
    [-30.0346, -51.2418], radius=500, color="#D55E00",
    fill=True, fill_opacity=0.12, tooltip="Raio ilustrativo: 500 m"
).add_to(mapa_formas)

folium.PolyLine(
    locais.iloc[:3][["latitude", "longitude"]].values.tolist(),
    color="#D55E00", weight=4, dash_array="8, 6",
    tooltip="Ligação ilustrativa — não é rota viária"
).add_to(mapa_formas)
mapa_formas

## 5. Mapas-base e controle de camadas

Um `FeatureGroup` reúne elementos que podem ser ligados e desligados juntos. `LayerControl` lista mapas-base e sobreposições; acrescente-o **depois** de adicionar as camadas.

As opções de fundo são alternativas entre si: OpenStreetMap ou somente os dados, sem mapa-base. As camadas de cultura e parques podem aparecer simultaneamente. Os provedores nomeados incluem a atribuição de autoria no mapa; mantenha esses créditos.

In [5]:
mapa_camadas = folium.Map(location=CENTRO_POA, zoom_start=13, tiles=None, height=500)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(mapa_camadas)
folium.FeatureGroup(name="Sem mapa-base", overlay=False, show=False).add_to(mapa_camadas)

for categoria, grupo in locais.groupby("categoria"):
    camada = folium.FeatureGroup(name=categoria)
    for local in grupo.itertuples(index=False):
        folium.Marker(
            [local.latitude, local.longitude], tooltip=local.nome,
            icon=folium.Icon(color=CORES[categoria])
        ).add_to(camada)
    camada.add_to(mapa_camadas)

folium.LayerControl(collapsed=False).add_to(mapa_camadas)
mapa_camadas

## 6. Muitos pontos: agrupamento de marcadores

Vamos gerar 180 observações **fictícias**, próximas aos locais anteriores, usando uma semente fixa. Elas não representam ocorrências reais, demanda ou visitas observadas.

`MarkerCluster` agrupa marcadores próximos na tela. Os números indicam quantos pontos estão em cada grupo; os agrupamentos mudam com o zoom. Clique em um grupo para explorar seus pontos.

In [6]:
rng = random.Random(42)
registros = []
for i in range(180):
    base = locais.iloc[i % len(locais)]
    registros.append({
        "id": i + 1,
        "latitude": float(base["latitude"]) + rng.gauss(0, 0.003),
        "longitude": float(base["longitude"]) + rng.gauss(0, 0.003),
    })
observacoes = pd.DataFrame(registros)

mapa_cluster = folium.Map(location=CENTRO_POA, zoom_start=13, height=500)
cluster = MarkerCluster(name="Observações fictícias").add_to(mapa_cluster)
for ponto in observacoes.itertuples(index=False):
    folium.Marker(
        [ponto.latitude, ponto.longitude], popup=f"Observação fictícia {ponto.id}"
    ).add_to(cluster)
folium.LayerControl().add_to(mapa_cluster)
mapa_cluster

## 7. Mapa de calor

`HeatMap` resume a concentração visual dos mesmos pontos. Cada ponto tem peso igual; `radius` e `blur` controlam a suavização em pixels.

As cores mostram **intensidade relativa**, não uma taxa por habitante nem uma densidade demográfica calibrada. O zoom e a suavização alteram a aparência. Compare com os agrupamentos antes de interpretar uma concentração.

**Experimente:** troque `radius=22` por `radius=40` e observe o que muda.

In [7]:
mapa_calor = folium.Map(
    location=CENTRO_POA, zoom_start=13, tiles="OpenStreetMap", height=500
)
HeatMap(
    observacoes[["latitude", "longitude"]].values.tolist(),
    name="Concentração de observações fictícias", radius=22, blur=18,
    min_opacity=0.3
).add_to(mapa_calor)
folium.LayerControl().add_to(mapa_calor)
mapa_calor

## 8. Dados reais: bairros de Porto Alegre

Este exemplo reutiliza os arquivos do notebook de mapas coropléticos:

- `dados/porto_alegre/RS_bairros_CD2022.gpkg`: malha de bairros do Censo 2022, IBGE;
- `dados/porto_alegre/populacao_bairros_porto_alegre_censo2022.csv`: população por bairro, ObservaPOA/Censo 2022.

Fontes: [IBGE](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/26565-malhas-de-setores-censitarios-divisoes-intramunicipais.html) e [ObservaPOA](https://prefeitura.poa.br/smpg/observapoa/mapas).

Selecionamos o município `4314902`, unimos fragmentos do mesmo bairro e associamos a população por nome normalizado. A verificação da junção evita produzir um mapa aparentemente completo com dados ausentes.

In [8]:
import re
import unicodedata
import geopandas as gpd

candidatos = [Path("dados/porto_alegre"), Path("notebooks/dados/porto_alegre")]
PASTA_DADOS = next((p for p in candidatos if (p / "RS_bairros_CD2022.gpkg").exists()), None)
if PASTA_DADOS is None:
    raise FileNotFoundError("Execute a partir da raiz IA001 ou da pasta notebooks, com os dados locais presentes.")

bairros_rs = gpd.read_file(PASTA_DADOS / "RS_bairros_CD2022.gpkg")
bairros = bairros_rs.loc[
    bairros_rs["CD_MUN"].astype(str) == "4314902",
    ["CD_BAIRRO", "NM_BAIRRO", "geometry"]
].dissolve(by=["CD_BAIRRO", "NM_BAIRRO"], as_index=False)

populacao = pd.read_csv(
    PASTA_DADOS / "populacao_bairros_porto_alegre_censo2022.csv",
    sep=";", skiprows=1, encoding="latin1", usecols=["Bairro", "Número"], dtype=str
).dropna(subset=["Bairro", "Número"])
populacao = populacao.loc[populacao["Bairro"] != "Total"].copy()
populacao["populacao"] = populacao["Número"].str.replace(".", "", regex=False).astype(int)

def normalizar_nome(texto):
    texto = unicodedata.normalize("NFKD", str(texto).replace("’", "'"))
    texto = texto.encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", texto.lower()).strip()

bairros["chave"] = bairros["NM_BAIRRO"].map(normalizar_nome)
populacao["chave"] = populacao["Bairro"].map(normalizar_nome)
bairros = bairros.merge(populacao[["chave", "populacao"]], on="chave", how="left", validate="one_to_one")
sem_dados = bairros.loc[bairros["populacao"].isna(), "NM_BAIRRO"].tolist()
if sem_dados:
    raise ValueError(f"Bairros sem população: {sem_dados}")
print(f"{len(bairros)} bairros; nenhum bairro sem população associada.")

94 bairros; nenhum bairro sem população associada.


### Área, densidade e coordenadas

Calculamos a área em `EPSG:31982`, uma projeção métrica adequada a Porto Alegre, e a densidade em habitantes por km². A área não deve ser calculada diretamente em graus de latitude/longitude.

Depois simplificamos apenas a geometria de exibição com tolerância de 25 m e convertemos para `EPSG:4326`, utilizado pelo GeoJSON no mapa web. As áreas e densidades usam a geometria anterior à simplificação.

In [9]:
bairros["area_km2"] = bairros.to_crs(31982).area / 1_000_000
if not bairros["area_km2"].gt(0).all():
    raise ValueError("Há bairros com área inválida.")
bairros["densidade"] = bairros["populacao"] / bairros["area_km2"]

bairros_web = bairros[["CD_BAIRRO", "NM_BAIRRO", "populacao", "area_km2", "densidade", "geometry"]].to_crs(31982).copy()
bairros_web["geometry"] = bairros_web.geometry.simplify(25, preserve_topology=True)
bairros_web = bairros_web.to_crs(4326)
bairros_web["CD_BAIRRO"] = bairros_web["CD_BAIRRO"].astype(str)
bairros_web[["NM_BAIRRO", "populacao", "area_km2", "densidade"]].head()

,NM_BAIRRO,populacao,area_km2,densidade
0,Medianeira,8749,1.397104,6262.240551
1,Praia de Belas,1522,2.615000,582.026766
2,Cidade Baixa,13014,0.759417,17136.838796
3,Menino-Deus,27961,2.298086,12167.082680
4,Farroupilha,774,0.600636,1288.633596


## 9. Mapa coroplético com legenda e tooltip

`Choropleth` associa os códigos da tabela (`columns`) aos códigos presentes no GeoJSON (`key_on`). A cor representa densidade, e não população total.

Usamos seis intervalos de mesma amplitude (`bins=6`). A classificação influencia o contraste entre bairros; a legenda informa os limites e a unidade. O tooltip mostra também população e área, permitindo interpretar o indicador.

**Experimente:** passe o mouse sobre os bairros e identifique um bairro grande com baixa densidade.

In [10]:
mapa_bairros = folium.Map(
    location=CENTRO_POA, zoom_start=11, tiles="OpenStreetMap",
    control_scale=True, height=600
)
coropletico = folium.Choropleth(
    geo_data=bairros_web.__geo_interface__,
    data=bairros_web, columns=["CD_BAIRRO", "densidade"],
    key_on="feature.properties.CD_BAIRRO",
    bins=6, fill_color="YlOrRd", fill_opacity=0.75,
    line_color="white", line_weight=0.6, line_opacity=0.8,
    nan_fill_color="lightgray", highlight=True,
    name="Densidade demográfica",
    legend_name="Porto Alegre — Censo 2022 — densidade (hab./km²)"
).add_to(mapa_bairros)
folium.GeoJsonTooltip(
    fields=["NM_BAIRRO", "populacao", "area_km2", "densidade"],
    aliases=["Bairro:", "População (2022):", "Área (km²):", "Densidade (hab./km²):"],
    localize=True, sticky=False
).add_to(coropletico.geojson)

# total_bounds usa longitude/latitude; fit_bounds recebe latitude/longitude.
oeste, sul, leste, norte = bairros_web.total_bounds
mapa_bairros.fit_bounds([[sul, oeste], [norte, leste]])
folium.LayerControl(collapsed=False).add_to(mapa_bairros)
mapa_bairros

## 10. Exportação para HTML

`save` gera um arquivo que pode ser aberto em um navegador sem executar Python. O arquivo inclui os dados geográficos, mas ainda precisa de internet para os recursos externos e os mapas-base.

A pasta de saída fica junto ao notebook. Se o link não abrir no ambiente Jupyter utilizado, abra o caminho impresso no navegador.

In [11]:
PASTA_SAIDA = PASTA_DADOS.parent.parent / "saidas_folium"
PASTA_SAIDA.mkdir(exist_ok=True)
ARQUIVO_HTML = PASTA_SAIDA / "porto_alegre_densidade_folium.html"
mapa_bairros.save(str(ARQUIVO_HTML))
print(ARQUIVO_HTML.resolve())
display(FileLink(str(ARQUIVO_HTML)))

/Users/joaocomba/Dropbox/Courses/IA001/notebooks/saidas_folium/porto_alegre_densidade_folium.html


/Users/joaocomba/Dropbox/Courses/IA001/notebooks/saidas_folium/porto_alegre_densidade_folium.html

## Para praticar

1. Adicione outro local à tabela e escolha sua categoria. Confirme a ordem das coordenadas.
2. Crie uma camada inicialmente oculta usando `FeatureGroup(show=False)`.
3. Compare o mapa de calor com dois valores de `radius`. Explique por que a imagem muda sem alterar os pontos.
4. No coroplético, compare `bins=4` com `bins=8`. Que diferenças ficam mais ou menos evidentes?
5. Exporte um dos mapas de marcadores para outro arquivo HTML.

### Qual recurso escolher?

| Pergunta | Recurso |
|---|---|
| Onde estão os locais? | `Marker`, `CircleMarker` |
| Como representar um raio ou uma ligação? | `Circle`, `PolyLine` |
| Como explorar muitos pontos individuais? | `MarkerCluster` |
| Onde há concentração visual de pontos? | `HeatMap` |
| Como comparar um indicador entre áreas? | `Choropleth` |
| Como alternar informações? | `FeatureGroup`, `LayerControl` |

### Referências

- [Guia oficial do Folium](https://python-visualization.github.io/folium/latest/user_guide.html)
- [Exemplos de coropléticos](https://python-visualization.github.io/folium/v0.16.0/user_guide/geojson/choropleth.html)
- [Leaflet](https://leafletjs.com/)